In [ ]:
# Python upgrade: Uncomment and run
# %pip upgrade python3

In [ ]:
%pip install -q -r requirements.txt

In [1]:
import datetime
import warnings

import pandas as pd
import numpy as np

# Data Preparation

# Overview

This notebook covers the cleaning and preparation of the datasets used to construct copulas to determine the monetary damage caused by severe weather events in two counties of Illinois, USA, namely Cook county (Cook IL) and Champaign county (Champaign IL). For this project, two data sources have been considered.

SHELDUS (Spatial Hazard Events and Losses Database for the United States) is a county-level hazard loss database available through Arizona State University. It was obtained under a data use agreement; for access information see the [SHELDUS website](https://sheldus.asu.edu) and the [End User License Agreement](https://sheldus.asu.edu/assets/END_USER_LICENSE_AGREEMENT.pdf). This research project maintains fair use of the datasets under $\S$ A 1.5 through the aggregation of at least two hazard types.

The National Oceanic and Atmospheric Administration (NOAA) Storm Events Database, maintained by NOAA's National Centers for Environmental Information (NCEI), contains records of severe weather phenomena including property damage estimates, as reported by the National Weather Service. It is publicly available on their [website](https://www.ncei.noaa.gov/stormevents/).

In [8]:
ckNOAAFull = pd.read_csv('cookNOAA.csv')
ckSheldusFull = pd.read_csv('cookSheldus.csv')
chNOAAFull = pd.read_csv('champaignNOAA.csv')
chSheldusFull = pd.read_csv('champaignSheldus.csv')

# Selection

The SHELDUS and NOAA datasets contain multiple columns, not all of which are relevant to this project. This section selects the pertinent variables and ensures consistency across both counties.

In [9]:
# Print out the columns of SHELDUS datasets to find any inconsistencies or differences in column names to work with

print(f"Cook SHELDUS Dataset: {ckSheldusFull.columns.tolist()}")
print(f"Champaign SHELDUS Dataset: {chSheldusFull.columns.tolist()}")

Cook SHELDUS Dataset: ['State Name', 'County Name', 'County FIPS', 'Hazard', 'Year', 'Month', 'CropDmg', 'CropDmg(ADJ 2020)', 'CropDmgPerCapita(ADJ 2020)', 'PropertyDmg', 'PropertyDmg(ADJ 2020)', 'PropertyDmgPerCapita(ADJ 2020)', 'Injuries', 'InjuriesPerCapita', 'Fatalities', 'FatalitiesPerCapita', 'Duration_Days', 'Fatalities_Duration', 'Injuries_Duration', 'Property_Damage_Duration', 'Crop_Damage_Duration', 'Records']
Champaign SHELDUS Dataset: ['State Name', 'County Name', 'County FIPS', 'Hazard', 'Year', 'Month', 'CropDmg', 'CropDmg(ADJ 2020)', 'CropDmgPerCapita(ADJ 2020)', 'PropertyDmg', 'PropertyDmg(ADJ 2020)', 'PropertyDmgPerCapita(ADJ 2020)', 'Injuries', 'InjuriesPerCapita', 'Fatalities', 'FatalitiesPerCapita', 'Duration_Days', 'Fatalities_Duration', 'Injuries_Duration', 'Property_Damage_Duration', 'Crop_Damage_Duration', 'Records']


In [10]:
# Print out the columns of NOAA datasets to find any inconsistencies or differences in column names to work with

print(f"Cook NOAA Dataset: {ckNOAAFull.columns.tolist()}")
print(f"Champaign NOAA Dataset: {chNOAAFull.columns.tolist()}")

Cook NOAA Dataset: ['Unnamed: 0', 'EVENT_ID', 'CZ_NAME_STR', 'BEGIN_LOCATION', 'BEGIN_DATE', 'BEGIN_TIME', 'EVENT_TYPE', 'MAGNITUDE', 'TOR_F_SCALE', 'DEATHS_DIRECT', 'INJURIES_DIRECT', 'DAMAGE_PROPERTY_NUM', 'DAMAGE_CROPS_NUM', 'STATE_ABBR', 'CZ_TIMEZONE', 'MAGNITUDE_TYPE', 'EPISODE_ID', 'CZ_TYPE', 'CZ_FIPS', 'WFO', 'INJURIES_INDIRECT', 'DEATHS_INDIRECT', 'SOURCE', 'FLOOD_CAUSE', 'TOR_LENGTH', 'TOR_WIDTH', 'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'END_RANGE', 'END_AZIMUTH', 'END_LOCATION', 'END_DATE', 'END_TIME', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON', 'EVENT_NARRATIVE', 'EPISODE_NARRATIVE', 'ABSOLUTE_ROWNUMBER']
Champaign NOAA Dataset: ['Unnamed: 0', 'EVENT_ID', 'CZ_NAME_STR', 'BEGIN_LOCATION', 'BEGIN_DATE', 'BEGIN_TIME', 'EVENT_TYPE', 'MAGNITUDE', 'TOR_F_SCALE', 'DEATHS_DIRECT', 'INJURIES_DIRECT', 'DAMAGE_PROPERTY_NUM', 'DAMAGE_CROPS_NUM', 'STATE_ABBR', 'CZ_TIMEZONE', 'MAGNITUDE_TYPE', 'EPISODE_ID', 'CZ_TYPE', 'CZ_FIPS', 'WFO', 'INJURIES_INDIRECT', 'DEATHS_INDIRECT', 'SOURCE', 'FLOOD_

Both NOAA datasets share identical column names, as do both SHELDUS datasets. Thus, no county-specific subsetting is needed.

SHELDUS datasets encompass a variety of weather events and their damages on crops, property and injuries & fatalities. NOAA datasets were obtained from the source website to include only thunderstorm wind events. To focus on only extreme wind weather events, the SHELDUS data is subset to retain only the columns related to property damage, time, and hazards. From the NOAA datasets, only the magnitude and time columns are retained.

In [11]:
colsSheldus = [
    "Hazard", "Year", "Month", 
    'PropertyDmg', 'PropertyDmg(ADJ 2020)', 'PropertyDmgPerCapita(ADJ 2020)'
    ]

ckSheldus = ckSheldusFull[colsSheldus]
chSheldus = chSheldusFull[colsSheldus]

SHELDUS records the property damages in US Dollars (USD). Alongside the raw value, it contains information about the value adjusted to 2022 for inflation, and per capita values

In [12]:
colsNOAA = ["BEGIN_DATE", "END_DATE", "MAGNITUDE"]

ckNOAA = ckNOAAFull[colsNOAA]
chNOAA = chNOAAFull[colsNOAA]

NOAA's NCEI records magnitudes in knots (kn) i.e. nautical miles per hour

# Creation

Currently, there are two datasets for each county: a SHELDUS dataset and a NOAA dataset. This section filters them and merges them based on the common element of time.

To narrow down the specific weather events concerned in this project, further subsets of the SHELDUS datasets are taken to include only thunderstorm wind events. 

In [14]:
ckSheldusStorms = ckSheldus[(ckSheldus['Hazard'] == "Severe Storm/Thunder Storm") | (ckSheldus['Hazard'] == "Wind")]
chSheldusStorms = chSheldus[(chSheldus['Hazard'] == "Severe Storm/Thunder Storm") | (chSheldus['Hazard'] == "Wind")]

## Establishing Time for each Datapoint

SHELDUS records data on a monthly basis. For uniformity, the year and month are combined to give a fixed month of occurrence in the form `YYYY-MM`

In [13]:
ckSheldus["When"] = pd.to_datetime(ckSheldus["Year"].astype(str) + "-" + ckSheldus["Month"].astype(str).str.zfill(2)).dt.to_period("M")
ckSheldus = ckSheldus.sort_values("When")

chSheldus["When"] = pd.to_datetime(chSheldus["Year"].astype(str) + "-" + chSheldus["Month"].astype(str).str.zfill(2)).dt.to_period("M")
chSheldus = chSheldus.sort_values("When")

In the event that multiple events occur in the same month, they are grouped together. The property damages are summed up.

In [15]:
ckSheldusStorms_grouped = ckSheldusStorms.groupby('When', as_index=False).agg(
    counts=('When', 'count'),
    PropertyDmg=('PropertyDmg', 'sum'),
    PropertyDmg_ADJ2020=('PropertyDmg(ADJ 2020)', 'sum'),
    PropertyDmgPerCapita_ADJ2020=('PropertyDmgPerCapita(ADJ 2020)', 'sum')
)

chSheldusStorms_grouped = chSheldusStorms.groupby('When', as_index=False).agg(
    counts=('When', 'count'),
    PropertyDmg=('PropertyDmg', 'sum'),
    PropertyDmg_ADJ2020=('PropertyDmg(ADJ 2020)', 'sum'),
    PropertyDmgPerCapita_ADJ2020=('PropertyDmgPerCapita(ADJ 2020)', 'sum')
)

NOAA records data by each thunderstorm wind event. Thus, it is necesssary to aggregate the datapoints by month before merging with the SHELDUS dataset, which is only possible if the begin & end dates of all weather events fall in the same month

In [16]:
ckNOAA[ckNOAA['BEGIN_DATE'] != ckNOAA['END_DATE']]

,BEGIN_DATE,END_DATE,MAGNITUDE


In [17]:
chNOAA[chNOAA['BEGIN_DATE'] != chNOAA['END_DATE']]

,BEGIN_DATE,END_DATE,MAGNITUDE
107,04/09/2001,04/10/2001,54.0


The NOAA dataset for Cook IL records no weather events across multiple days. The one for Champaign IL records an event that spans two days, but lies within the same month. Thus, NOAA datapoints can be merged on the basis of the month of occurrence. Without loss of generality, the month stated in the beginning of the occurrence is taken as the determining month.

In [18]:
ckNOAA['When'] = pd.to_datetime(ckNOAA['BEGIN_DATE']).dt.to_period('M')
chNOAA['When'] = pd.to_datetime(chNOAA['BEGIN_DATE']).dt.to_period('M')

While merging events of differing magnitudes in the same month, the maximum of the magnitudes is considered to be the representative magnitude for weather events having occurred that month.

In [19]:
ckNOAA_grouped = ckNOAA.groupby('When', as_index=False).agg(
    counts=('When', 'count'),
    MAGNITUDE=('MAGNITUDE', 'max')
)

In [20]:
chNOAA_grouped = chNOAA.groupby('When', as_index=False).agg(
    counts=('When', 'count'),
    MAGNITUDE=('MAGNITUDE', 'max')
)

## Joining the Datasets

The individual datasets by county and source are now ready to be joined to create representative datasets for each county

In [21]:
ckNOAAToJoin = ckNOAA_grouped[['MAGNITUDE', 'When']]
ckSheldusToJoin = ckSheldusStorms_grouped[['When', 'PropertyDmg', 'PropertyDmg_ADJ2020','PropertyDmgPerCapita_ADJ2020']]
chNOAAToJoin = chNOAA_grouped[['MAGNITUDE', 'When']]
chSheldusToJoin = chSheldusStorms_grouped[['When', 'PropertyDmg', 'PropertyDmg_ADJ2020','PropertyDmgPerCapita_ADJ2020']]

An outer join is used to join the datasets to retain most of the information possible

In [22]:
ckJoined = pd.merge(ckSheldusToJoin, ckNOAAToJoin, on='When', how='outer')
chJoined = pd.merge(chSheldusToJoin, chNOAAToJoin, on='When', how='outer')

The column names are made uniform for ease during this project.

In [23]:
ckJoined.rename(columns={'MAGNITUDE': 'Magnitude',
                         'PropertyDmg':'PropDmg',
                         'PropertyDmg_ADJ2020':'PropDmg (adj. 2020)',
                         'PropertyDmgPerCapita_ADJ2020':'PropDmgPerCapita (adj. 2020)'},
                inplace=True)
chJoined.rename(columns={'MAGNITUDE': 'Magnitude',
                         'PropertyDmg':'PropDmg',
                         'PropertyDmg_ADJ2020':'PropDmg (adj. 2020)',
                         'PropertyDmgPerCapita_ADJ2020':'PropDmgPerCapita (adj. 2020)'},
                inplace=True)

# Coherence

Inconsistencies might arise through the creation of the merged dataset via an outer join. This section assesses the reasonability of the data.

## Null Values

Property damages are assessed to check discrepancies among the 3 types of property damages: raw value, value adjusted to 2020 and the per capita value adjusted to 2020.

In [24]:
display(pd.crosstab(
    ckJoined['PropDmg'].isna(),
    [ckJoined['PropDmg (adj. 2020)'].isna(), ckJoined['PropDmgPerCapita (adj. 2020)'].isna()],
    rownames=['PropDmg'],
    colnames=['PropDmg (adj. 2020)', 'PropDmgPerCapita (adj. 2020)']
))

PropDmg (adj. 2020),False,True
PropDmgPerCapita (adj. 2020),False,True
PropDmg,,
False,255,0
True,0,77


In [25]:
display(pd.crosstab(
    chJoined['PropDmg'].isna(),
    [chJoined['PropDmg (adj. 2020)'].isna(), chJoined['PropDmgPerCapita (adj. 2020)'].isna()],
    rownames=['PropDmg'],
    colnames=['PropDmg (adj. 2020)', 'PropDmgPerCapita (adj. 2020)']
))

PropDmg (adj. 2020),False,True
PropDmgPerCapita (adj. 2020),False,True
PropDmg,,
False,159,0
True,0,64


In each of the datapoints of the datasets created, either all 3 of the variables are null or filled.

Datapoints with `NaN` values in all columns except the month are redundant.

In [18]:
display(pd.crosstab(ckJoined['PropDmg'].isna(), ckJoined['Magnitude'].isna()))
display(pd.crosstab(ckJoined['PropDmg (adj. 2020)'].isna(), ckJoined['Magnitude'].isna()))
display(pd.crosstab(ckJoined['PropDmgPerCapita (adj. 2020)'].isna(), ckJoined['Magnitude'].isna()))

Magnitude,False,True
PropDmg,,
False,135,120
True,76,1


Magnitude,False,True
PropDmg (adj. 2020),,
False,135,120
True,76,1


Magnitude,False,True
PropDmgPerCapita (adj. 2020),,
False,135,120
True,76,1


In [19]:
display(pd.crosstab(chJoined['PropDmg'].isna(), chJoined['Magnitude'].isna()))
display(pd.crosstab(chJoined['PropDmg (adj. 2020)'].isna(), chJoined['Magnitude'].isna()))
display(pd.crosstab(chJoined['PropDmgPerCapita (adj. 2020)'].isna(), chJoined['Magnitude'].isna()))

Magnitude,False,True
PropDmg,,
False,89,70
True,57,7


Magnitude,False,True
PropDmg (adj. 2020),,
False,89,70
True,57,7


Magnitude,False,True
PropDmgPerCapita (adj. 2020),,
False,89,70
True,57,7


They provide no information useful to the project, so they are removed.

In [20]:
ckJoined = ckJoined.dropna(subset=['PropDmg', 'Magnitude'], how='all')
chJoined = chJoined.dropna(subset=['PropDmg', 'Magnitude'], how='all')

## `Magnitude` and Property Damage Discrepancy

The dataset contains datapoints where the `Magnitude` of the weather event was not recorded and the property damage was reported to be 0.

In [21]:
display(pd.crosstab(ckJoined['PropDmg'] == 0, ckJoined['Magnitude'].isna()),
        pd.crosstab(chJoined['PropDmg'] == 0, chJoined['Magnitude'].isna()))

Magnitude,False,True
PropDmg,,
False,203,81
True,8,39


Magnitude,False,True
PropDmg,,
False,143,65
True,3,5


This is the consequence of the outer join on datapoints in SHELDUS where property damages were recorded, but such a weather event and its magnitude were not recorded in the corresponding month of the NOAA dataset. These discrepancies are excluded to avoid hallucinations in the project.

In [22]:
chJoined = chJoined[~((chJoined['Magnitude'].isna()) & (chJoined['PropDmg'] == 0))]
ckJoined = ckJoined[~((ckJoined['Magnitude'].isna()) & (ckJoined['PropDmg'] == 0))]

The dataset also contains datapoints where `Magnitude` of the weather event was reported to be 0 kn and some non-zero property damages were recorded.


In [23]:
display(pd.crosstab(ckJoined['PropDmg'] > 0, ckJoined['Magnitude'] == 0),
        pd.crosstab(chJoined['PropDmg'] > 0, chJoined['Magnitude'] == 0))

Magnitude,False,True
PropDmg,,
False,68,16
True,180,28


Magnitude,False,True
PropDmg,,
False,53,7
True,131,20


In this case, there were definitive thunderstorm wind events that were captured in both the SHELDUS and NOAA datasets. Although the property damages could have potentially been sustained because of another weather event that occurred in conjunction with the thunderstorm wind event, these datapoints are present in both the *filtered* datasets, and discarding them would lose relevant observations. Thus the `Magnitude` values in such cases are set to `NaN` and will be imputed in the next part of this project.

In [24]:
ckJoined.loc[(ckJoined['PropDmg'] > 0) & (ckJoined['Magnitude'] == 0), 'Magnitude'] = np.nan
chJoined.loc[(chJoined['PropDmg'] > 0) & (chJoined['Magnitude'] == 0), 'Magnitude'] = np.nan

# Final Datasets

The final datasets are now ready for analysis. There are two datasets, one for Cook county and one for Champaign county. This section saves them for ease of usage in the rest of this project

The datasets are first sorted by month

In [25]:
ckJoined.sort_values('When', inplace=True)
chJoined.sort_values('When', inplace=True)

They are then re-indexed

In [26]:
chJoined = chJoined.reset_index(drop=True)
ckJoined = ckJoined.reset_index(drop=True)

The datasets are finally saved and ready for usage

In [27]:
chJoined.to_csv('champaignIL.csv', index=False)
ckJoined.to_csv('cookIL.csv', index=False)

# Fin